# VIX smiles in the one-factor Bergomi model

This notebook reproduces the Bergomi figures of the paper:

- Figure 1: single kernel (Section 2.3).
- Figures 3, 4, 8, 9, 12 and 13, and the top row of Figure 7: Scenarios 1 and 2
  with two kernels (Section 3 and Appendix C).

It calls the same functions as `scripts/reproduce_single_figures.py` and
`scripts/reproduce_mixed_figures.py`, so it gives the same numbers. Run it from
the repository root. It takes a few minutes, mostly for the reference prices.

The kernel is `K(u, t) = omega * exp(-kappa * (u - t))`, the VIX window is
`Delta = 30/365` and `xi0 = 0.24**2`.

In [ ]:
import sys

sys.path.insert(0, "scripts")

import numpy as np

import common as c
import reproduce_mixed_figures as mixed
import reproduce_single_figures as single

%matplotlib inline

args = c.default_args()  # the paper's reference settings
confidence = c.confidence_factor(args.replicates)

## Single kernel (Figure 1)

`(omega, kappa) = (2, 0.25)` in the top row and `(8, 10)` in the bottom row.
*Approx* is `implied_vol_vix_approx`, the implied volatility of the weak price
approximation. *Expansion* is `implied_vol_vix_expansion`, the closed-form
implied volatility expansion (Theorem 2). The reference uses quadrature: 256
Gauss-Legendre nodes in the delivery date and adaptive quadrature in the
Gaussian factor.

In [ ]:
single_rows = single.calculate(
    args, cases=[("standard", 1), ("standard", 2)], validate=False
)
single.plot(single_rows, confidence, save=False)

### One smile, step by step

The cell above hides the loop over scenarios and maturities. Here is the
3-month smile of Bergomi case 1, computed directly. The strikes are
`F_P * exp(m)`, where `F_P` is the weak approximation of the VIX futures and
`m` runs over `c.MONEYNESS`.

In [ ]:
model = c.single_model("standard", 1)
T = 3 / 12
n = c.COEFFICIENT_NODES

forward = model.price_vix_fut_approx(T=T, order=3, n_quad=n)
strikes = forward * np.exp(c.MONEYNESS)

approx = model.implied_vol_vix_approx(k=c.MONEYNESS, T=T, order=3, n_quad=n)
expansion = model.implied_vol_vix_expansion(k=c.MONEYNESS, T=T, order=1, n_quad=n)
reference = c.reference("standard", (model, model), T, 0.5, strikes, args)

print(" log-moneyness  reference    approx  expansion")
for m, ref, a, e in zip(c.MONEYNESS, reference["iv"], approx, expansion):
    print(f"{m:14.3f} {100 * ref:10.4f} {100 * a:9.4f} {100 * e:10.4f}")

## Two kernels: Scenarios 1 and 2

The forward variance is a mixture `lambda * exp(Z_1) + (1 - lambda) * exp(Z_2)`
of two exponential kernels (Section 3). For each scenario and maturity:

- *Approx* inverts the weak prices of `mixed_iv.mixed_proxy_prices`.
- *Expansion* in Figure 7 is `Sigma0 + Sigma1` from
  `mixed_iv.proxy_smile_expansion` (Section 3.3).
- *Expansion* in Figures 3 and 4 is the explicit Hermite
  expansion (60) around component `j`, from `mixed_iv.hermite_expansion`
  (Section 3.2). The extended expansion (61) and the Hermite call price come
  from the same call.

This cell also takes a minute or two.

In [ ]:
mixed_rows = mixed.calculate(args, scenarios=(1, 2), validate=False)

### Scenario 1

In order: Figure 7 (top left panel),
Figure 3 (smile and error for each
component shown in the paper), Figure 12
(extended expansion) and Figure 8 (call prices).

In [ ]:
mixed.plot([r for r in mixed_rows if r["scenario"] == 1], confidence, save=False)

### Scenario 2

In order: Figure 7 (top right panel),
Figure 4 (smile and error for each
component shown in the paper), Figure 13
(extended expansion) and Figure 9 (call prices).

In [ ]:
mixed.plot([r for r in mixed_rows if r["scenario"] == 2], confidence, save=False)